# Create Architecture Tables

Compute the number of trainable head parameters used for binary classification and binary segmentation with every model backbone configured under `src/ultrabench/training/config/model/`, save the results to CSV files, and render booktabs-style LaTeX tables in the same visual style as `create_results_tables.ipynb`.

Each backbone is instantiated directly (without loading a checkpoint) and the heads are built with the `DownstreamPipeline._build_head` factory method used by the training pipeline to ensure that the reported counts are accurate.

The generated tables require `\usepackage{booktabs}`, `\usepackage{siunitx}` and `\usepackage{graphicx}` in the document preamble.

In [ ]:
from pathlib import Path
from unittest.mock import patch

import pandas as pd
from omegaconf import OmegaConf

from ultrabench.models.deblurmim.deblurmim_backbone import DeblurMIMBackbone
from ultrabench.models.dinov3.dinov3_backbone import (
    DINOv3ConvNeXtBackbone,
    DINOv3ViTBackbone,
)
from ultrabench.models.echocare.echocare_backbone import EchoCareBackbone
from ultrabench.models.openus.openus_backbone import OpenUSBackbone
from ultrabench.models.samus.samus_backbone import SAMUSBackbone
from ultrabench.models.tinyusfm.tinyusfm_backbone import TinyUSFMBackbone
from ultrabench.models.ultrafedfm.ultrafedfm_backbone import UltraFedFMBackbone
from ultrabench.models.ultrasam.ultrasam_backbone import UltraSAMBackbone
from ultrabench.models.urfm.urfm_backbone import URFMBackbone
from ultrabench.models.usfm.usfm_backbone import USFMBackbone
from ultrabench.models.usfmae.usfmae_backbone import USFMAEBackbone
from ultrabench.training.heads.seg_heads import DECODER_COMPATIBILITY
from ultrabench.training.setup_ptl import DownstreamPipeline

MODEL_DISPLAY_NAMES = {
    "deblurmim_convmae": "Deblurring ConvMAE",
    "deblurmim_mae": "Deblurring MAE",
    "dinov3_convnext_base": "DINOv3 ConvNeXt-B",
    "dinov3_convnext_large": "DINOv3 ConvNeXt-L",
    "dinov3_vitb16": "DINOv3 ViT-B",
    "dinov3_vitl16": "DINOv3 ViT-L",
    "echocare": "EchoCare",
    "openus": "OpenUS",
    "samus": "SAMUS",
    "tinyusfm": "TinyUSFM",
    "ultrafedfm": "UltraFedFM",
    "ultrasam": "UltraSAM",
    "urfm_base": "URFM-B",
    "urfm_large": "URFM-L",
    "usfm": "USFM",
    "usfmae": "USF-MAE",
}

OUTPUT_TYPE_DISPLAY = {
    "patch_tokens": "Patch Tokens",
    "spatial_map": "Spatial Map",
    "multi_scale": "Multi-Scale",
}

DECODER_DISPLAY_NAMES = {
    "upsampler": "Upsampler",
    "segformer": "SegFormer",
}

CONFIG_DIR = Path("../src/ultrabench/training/config")
MODEL_CONFIG_DIR = CONFIG_DIR / "model"
HEAD_CONFIG_DIR = CONFIG_DIR / "head"

NUM_CLASSES = 2  # Binary classification / segmentation

# Arbitrary -- none of the three segmentation decoders' parameter counts
# depend on the output image size, only their cosmetic final
# `F.interpolate` calls do.
IMAGE_SIZE = 256

# The siunitx setup for the parameter-count columns. `group-digits` and
# `group-separator` render large integers with thousands separators (e.g.
# 4,936,706).
SISETUP_OPTIONS = [
    "group-digits = true",
    "group-separator = {,}",
    "table-format = 8.0",
]

TABLE_FONT_SIZE = r"\fontsize{10}{12}\selectfont"

## Instantiate Backbones

Each backbone is instantiated with checkpoint loading mocked out.

In [ ]:
_BACKBONE_SPECS: list[tuple[str, type, dict]] = [
    (
        "deblurmim_convmae",
        DeblurMIMBackbone,
        {
            "checkpoint_path": "dummy",
            "output_intermediate_features": True,
            "model_variant": "deblurmim_convmae_base",
        },
    ),
    (
        "deblurmim_mae",
        DeblurMIMBackbone,
        {
            "checkpoint_path": "dummy",
            "output_intermediate_features": True,
            "model_variant": "deblurmim_mae_base_patch16",
        },
    ),
    (
        "dinov3_convnext_base",
        DINOv3ConvNeXtBackbone,
        {
            "checkpoint_path": "dummy",
            "repo_dir": "dummy",
            "output_intermediate_features": True,
            "model_variant": "dinov3_convnext_base",
        },
    ),
    (
        "dinov3_convnext_large",
        DINOv3ConvNeXtBackbone,
        {
            "checkpoint_path": "dummy",
            "repo_dir": "dummy",
            "output_intermediate_features": True,
            "model_variant": "dinov3_convnext_large",
        },
    ),
    (
        "dinov3_vitb16",
        DINOv3ViTBackbone,
        {
            "checkpoint_path": "dummy",
            "repo_dir": "dummy",
            "output_intermediate_features": True,
            "model_variant": "dinov3_vitb16",
        },
    ),
    (
        "dinov3_vitl16",
        DINOv3ViTBackbone,
        {
            "checkpoint_path": "dummy",
            "repo_dir": "dummy",
            "output_intermediate_features": True,
            "model_variant": "dinov3_vitl16",
        },
    ),
    (
        "echocare",
        EchoCareBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "openus",
        OpenUSBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "samus",
        SAMUSBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": False},
    ),
    (
        "tinyusfm",
        TinyUSFMBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "ultrafedfm",
        UltraFedFMBackbone,
        {
            "checkpoint_path": "dummy",
            "output_intermediate_features": True,
            "model_variant": "ultrafedfm_vit_base_patch16",
        },
    ),
    (
        "ultrasam",
        UltraSAMBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "urfm_base",
        URFMBackbone,
        {
            "checkpoint_path": "dummy",
            "model_variant": "urfm_mae_vit_base_patch16",
            "output_intermediate_features": True,
        },
    ),
    (
        "urfm_large",
        URFMBackbone,
        {
            "checkpoint_path": "dummy",
            "model_variant": "urfm_mae_vit_large_patch16",
            "output_intermediate_features": True,
        },
    ),
    (
        "usfm",
        USFMBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "usfmae",
        USFMAEBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
]

model_config_keys = {path.stem for path in MODEL_CONFIG_DIR.glob("*.yaml")}
spec_keys = {model_key for model_key, _, _ in _BACKBONE_SPECS}
assert model_config_keys == spec_keys, (
    f"_BACKBONE_SPECS is out of sync with {MODEL_CONFIG_DIR}: "
    f"missing={model_config_keys - spec_keys}, extra={spec_keys - model_config_keys}"
)

backbones = {}
for model_key, cls, kwargs in _BACKBONE_SPECS:
    with patch.object(cls, "_load_checkpoint", lambda self: None):
        backbones[model_key] = cls(**kwargs)

len(backbones)

## Load Head Configurations

Load the four head configs directly from `training/config/head/` and instantiate the `DownstreamPipeline`, whose `_build_head` method is the same factory the real training pipeline uses.

In [ ]:
HEAD_CONFIGS = {
    name: OmegaConf.load(HEAD_CONFIG_DIR / f"{name}.yaml")
    for name in ["linear_classifier", "upsampler", "segformer"]
}

pipeline = DownstreamPipeline()

## Compute Classification Head Parameters

Build the `linear_classifier` head for every backbone and count its trainable
parameters.

In [ ]:
def count_params(head) -> int:
    """Count trainable parameters in a head module.

    Args:
        head: A head module returned by `DownstreamPipeline._build_head`.

    Returns:
        The total number of trainable parameters in the head.
    """
    return sum(p.numel() for p in head.parameters())


classification_rows = []
for model_key, backbone in backbones.items():
    head, _ = pipeline._build_head(
        backbone, HEAD_CONFIGS["linear_classifier"], NUM_CLASSES, "classification"
    )
    classification_rows.append(
        {
            "model": model_key,
            "feature_dim": backbone.feature_dim,
            "num_params": count_params(head),
        }
    )

classification_df = pd.DataFrame(classification_rows)
classification_df.to_csv("classification_architecture_params.csv", index=False)
classification_df

## Compute Segmentation Head Parameters

Build the segmentation decoder head used with each backbone (based on their `output_type`, see per `DECODER_COMPATIBILITY`) and count its trainable parameters.

In [ ]:
segmentation_rows = []
for model_key, backbone in backbones.items():
    for decoder_type, compatible_output_types in DECODER_COMPATIBILITY.items():
        if decoder_type == "setr_mla":
            # Never used. patch_tokens backbones are evaluated with the
            # upsampler decoder instead.
            continue
        if backbone.output_type not in compatible_output_types:
            continue
        head, _ = pipeline._build_head(
            backbone,
            HEAD_CONFIGS[decoder_type],
            NUM_CLASSES,
            "segmentation",
            image_size=IMAGE_SIZE,
        )
        segmentation_rows.append(
            {
                "model": model_key,
                "output_type": backbone.output_type,
                "decoder_type": decoder_type,
                "num_params": count_params(head),
            }
        )

segmentation_df = pd.DataFrame(segmentation_rows)
segmentation_df.to_csv("segmentation_architecture_params.csv", index=False)
segmentation_df

## Build the LaTeX Tables

Render each results table as a booktabs-style LaTeX table environment, following the same visual style (font size, column separation, `siunitx` setup) as `create_results_tables.ipynb`.

In [ ]:
def format_int(value: int) -> str:
    """Format an integer for a siunitx `S` column.

    Args:
        value: The integer value to format.

    Returns:
        The value as a plain digit string; siunitx applies the thousands
        separator via `group-digits` in `\\sisetup`.
    """
    return str(int(value))


def to_booktabs_latex(
    frame: pd.DataFrame, column_format: str, caption: str, label: str
) -> str:
    """Render a table of formatted cells as a booktabs LaTeX table.

    Args:
        frame: A DataFrame of already-formatted cell strings.
        column_format: The tabular column format string, e.g. `"lS"`.
        caption: The caption of the table.
        label: The label used to reference the table.

    Returns:
        The complete LaTeX source of the table.
    """
    # `names=True` is deliberately omitted: with pandas 3.0.0 (pinned in
    # pyproject.toml), passing it together with `axis="index"` in the same call
    # silently no-ops the hide, leaving the row index in the rendered table.
    tabular = frame.style.hide(axis="index").to_latex(
        hrules=True, siunitx=True, column_format=column_format
    )
    sisetup = "\n".join(
        [r"\sisetup{", *(f"  {option}," for option in SISETUP_OPTIONS), "}"]
    )

    # These tables are narrow enough to fit the text width unscaled, unlike the
    # per-task results tables in `create_results_tables.ipynb`, so no
    # `\resizebox` wrapper is used here.
    return "\n".join(
        [
            r"\begin{table}[t]",
            r"\centering",
            rf"\caption{{{caption}}}",
            rf"\label{{{label}}}",
            TABLE_FONT_SIZE,
            r"\setlength{\tabcolsep}{3pt}",
            sisetup,
            tabular.strip(),
            r"\end{table}",
            "",
        ]
    )

## Generate the Tables

Write each table to a `.tex` file alongside this notebook and print its source.

In [ ]:
classification_table = pd.DataFrame(
    {
        "Model": classification_df["model"].map(MODEL_DISPLAY_NAMES),
        "Feature Dimension": classification_df["feature_dim"].map(format_int),
        "Trainable Parameters": classification_df["num_params"].map(format_int),
    }
)
classification_latex = to_booktabs_latex(
    classification_table,
    column_format="lSS",
    caption=(
        "The number of trainable parameters in the linear classification head "
        "added to each model backbone for a binary classification task."
    ),
    label="tab:classification_architecture",
)
Path("classification_architecture_params.tex").write_text(classification_latex)
print(classification_latex)

In [ ]:
segmentation_table = pd.DataFrame(
    {
        "Model": segmentation_df["model"].map(MODEL_DISPLAY_NAMES),
        "Output Type": segmentation_df["output_type"].map(OUTPUT_TYPE_DISPLAY),
        "Head Architecture": segmentation_df["decoder_type"].map(DECODER_DISPLAY_NAMES),
        "Trainable Parameters": segmentation_df["num_params"].map(format_int),
    }
)
segmentation_latex = to_booktabs_latex(
    segmentation_table,
    column_format="lllS",
    caption=(
        "The number of trainable parameters in each segmentation decoder head "
        "compatible with each model backbone's output type, for a binary "
        "segmentation task."
    ),
    label="tab:segmentation_architecture",
)
Path("segmentation_architecture_params.tex").write_text(segmentation_latex)
print(segmentation_latex)